In [ ]:
import os
os.chdir("/data/local/mvaquerizo/transcriptomica/")

### Libraries

In [ ]:
import sklearn as sk
import pandas as pd
import statistics as st
import numpy as np
from scripts.Funciones.Funciones import corr_loop, pca_plot, accuracy_models, prob_to_clase, performance_modelo, rendimiento_train, rendimiento_test
import scipy.stats as sc
import igraph as ig
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold, KFold, RepeatedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from multiprocessing import Pool
import pickle
import random


### Load data

In [ ]:
data = pd.read_csv('results/06_exploratory_and_DEA_limma/conteos_finales.csv', index_col=0)

In [ ]:
ed = pd.read_csv('results/06_exploratory_and_DEA_limma/metadata.csv', index_col=0)

### Clasificación
Separamos las muestras en envejecidas (E) y rejuvenecidas (R) en base a los datos epigenéticos

In [ ]:
ed.index.tolist() == data.columns.tolist()

In [ ]:
classification = pd.DataFrame({'sample': ed.index.tolist(), 
                              'class': list(ed.g_acc),
                              'class_numeric': ed.g_acc.map({'R':0, 'E':1})},
                              index = ed.index.tolist())
classification

### Selección de genes
Nos queremos quedar con aquellos genes correlacionados con la clase, pero que estén poco correlacionados entre sí para evitar colinealidad

In [ ]:
# Cálculo correlación gen - clase 
corr1 = pd.DataFrame()
for i in range(0,data.shape[0]):
    res = sc.spearmanr(data.iloc[i,:], classification.loc[:,'class_numeric'])
    res2 = pd.DataFrame.from_dict({'R': [res.statistic], 'p_value': [res.pvalue]})
    corr1 = pd.concat([corr1, res2])

In [ ]:
# Nos quedamos solo con aquellos genes con correlación positiva con la clase
corr1.index = data.index
corr1['Genes'] = corr1.index
corr_genes = corr1[corr1['p_value']<=0.05].index.to_list()
subdat = data.loc[corr_genes, :]
subdat.shape

In [ ]:
# Calculamos la correlación gen a gen 
index_loop = []
for i in range(0,subdat.shape[0]):
    for j in range(i+1, subdat.shape[0]):
        index_loop.append((subdat,i,j))

with Pool() as pool:
    results = pool.map(corr_loop, index_loop)
corr2 = pd.concat(results)
corr2

In [ ]:
# Cogemos aquellos pares de genes altamente correlacionados entre ellos (R > 0.75) y generamos grafos de conectividad 
g = ig.Graph.TupleList(corr2[(corr2['p_value']<0.05) & (abs(corr2['R'])>0.75)].loc[:, ['Gene1', "Gene2"]].itertuples(index=False))
components = g.connected_components(mode='weak')
# Establecemos si están conectados entre sí o no
toselection = corr1.loc[g.get_vertex_dataframe()['name'],:]
toselection['membership'] = components.membership
len(toselection)

In [ ]:
# Seleccionamos los genes más conectados de cada grupo 
selected_genes = []
for i in set(toselection['membership']):
    values = toselection.loc[toselection['membership'] == i]
    selected_genes.append(values.iloc[pd.Series(abs(values['R']).tolist()).idxmax(), :].name)
len(selected_genes)

In [ ]:
# Al final nos quedamos con todos los genes correlacionados con la clase (R_clase < 0.05) 
# menos aquellos correlacionados entre ellos salvo los seleccionados 
remove_genes = [gene for gene in toselection.index.to_list() if gene not in selected_genes]
corr1_final = corr1.query('Genes not in @remove_genes and p_value < 0.05')
len(corr1_final.index)

In [ ]:
corr1_final['R_abs'] = abs(corr1_final.R)
corr1_final.sort_values('R_abs', ascending=False, inplace=True)
corr1_final

In [ ]:
# Guardamos los genes seleccionados 
with open('results/corr1_final.pickel', 'wb') as file: 
    pickle.dump(corr1_final, file)

In [ ]:
data_filt = data.loc[corr1_final.index, :].T
data_filt

In [ ]:
# Mezclamos las muestras para que no estén separadas E y R 
sample = list(data_filt.index)
random.shuffle(sample)
data_filt = data_filt.loc[sample, :]
ed_filt = ed.loc[sample, :]

In [ ]:
ed_filt.index.to_list() == data_filt.index.to_list()

In [ ]:
y = list(ed_filt.g_acc)
y

In [ ]:
# Guardamos los datos 
with open('results/datos_filtrados.pickel', 'wb') as file: 
    pickle.dump(data_filt, file)
    pickle.dump(ed_filt, file)
    pickle.dump(y, file)

### Cross-validation
Generamos la separación de las muestras del train original en train-test para el entrenamiento y validación interna de los modelos. Tenemos que generar en total la división train-test 5 fold 10 veces, por lo que al final tenemos 50 conjuntos de datos

In [ ]:
# Vamos generando las divisones y guardamos los índices de las muestras en un diccionario de diccionarios.
kf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=123)

cross_val = {}
aux = {}
time = 1
fold = 1

for train, test in kf.split(data_filt):
    aux[fold] = {'Train': train, 'Test': test}
    if fold < 5: 
        fold += 1
    else: 
        cross_val[time] = aux
        aux = {}
        fold = 1
        time += 1
    
cross_val	

In [ ]:
# Guardamos el cross_val en un archivo para usar siempre la misma separación de muestras
with open('results/cross_val.pickel', 'wb') as file: 
    pickle.dump(cross_val, file)

In [ ]:
results = []

n_genes = (list(range(1, 51)) +
           list(range(55, 101, 5)) +
           list(range(110, 201, 10)) +
           list(range(220, data_filt.shape[1] + 1, 20)) +
           [data_filt.shape[1]]
          )
                   
n_genes = sorted(set(n_genes))

for i in n_genes:
    print(i)
    sel_genes = corr1_final.iloc[:i, :]['Genes'].to_list()
    sel_data = data_filt.loc[:, sel_genes]

    acc_svm, acc_rf, acc_knn = accuracy_models(cross_val, sel_data, y)

    results.append({'genes': i, 'SVM': acc_svm, 'Rf': acc_rf, 'kNN': acc_knn})

accuracy = pd.DataFrame(results)
accuracy

In [ ]:
# Guardamos la información de accuracy en un csv 
accuracy.to_csv('results/accuracy_v2.csv', index=False)

In [ ]:
accuracy = pd.read_csv('results/accuracy.csv', index_col=0)
n_genes = (list(range(1, 51)) +
           list(range(55, 101, 5)) +
           list(range(110, 201, 10)) +
           list(range(220, data_filt.shape[1] + 1, 20)) +
           [data_filt.shape[1]]
          )
                   
n_genes = sorted(set(n_genes))

In [ ]:
fig, ax = plt.subplots()
plt.plot(n_genes, accuracy.svm, color = '#8AB6E9')
plt.plot(n_genes, accuracy.rf, color = '#F1788D')
plt.plot(n_genes, accuracy.knn, color = '#86C157')
plt.legend(['SVM', 'RF', 'kNN'])
plt.show()
#fig.savefig('results/plot_precision.jpg', dpi=500)
plt.close(fig)

In [ ]:
# Sacamos el número de genes y modelo que obtiene el mayor accuracy
n = accuracy.stack().idxmax()[0]
n

In [ ]:
# Sacamos los genes de la firma y los guardamos en un fichero de texto 
sel_genes = corr1_final.iloc[:n, :]['Genes'].to_list()
with open("results/08_gene_signature/genes.txt", "w") as file:
    for gen in sel_genes: 
        file.write(f'{gen}\n')